# 01 · Coleta e limpeza

**Objetivo:** carregar os focos de queimadas do INPE (2015 a 2025, satélite de referência), filtrar os biomas Amazônia, Cerrado e Pantanal e gerar a série mensal por bioma.

Os dados vêm dos arquivos anuais do INPE (`focos_br_ref_ANO.csv`) e ficam em `data/raw/`.


In [1]:
import sys; sys.path.append('..')
from src import data_loader, features
from src.config import PROCESSED_DIR

In [ ]:
# data_loader.download_all() # baixa os CSVs de 2015 a 2025 (ainda não testado)

df = data_loader.load_raw()
df.head()

,id_bdq,foco_id,lat,lon,data_pas,pais,estado,municipio,bioma,datahora
0,157098753,57d0e58b-5abb-11e8-911c-28924ad12c5c,-16.291,-43.863,2015-01-05 16:38:00,Brasil,MINAS GERAIS,MONTES CLAROS,Cerrado,2015-01-05 16:38:00
1,157194528,8441c3cd-5aba-11e8-911c-28924ad12c5c,-10.285,-59.470,2015-01-06 17:23:00,Brasil,MATO GROSSO,ARIPUANÃ,Amazônia,2015-01-06 17:23:00
2,157194916,8441c413-5aba-11e8-911c-28924ad12c5c,3.709,-61.079,2015-01-06 17:27:00,Brasil,RORAIMA,AMAJARI,Amazônia,2015-01-06 17:27:00
3,157194529,8441c3ce-5aba-11e8-911c-28924ad12c5c,-11.154,-54.248,2015-01-06 17:23:00,Brasil,MATO GROSSO,MARCELÂNDIA,Amazônia,2015-01-06 17:23:00
4,157194530,8441c3cf-5aba-11e8-911c-28924ad12c5c,-11.378,-54.711,2015-01-06 17:23:00,Brasil,MATO GROSSO,CLÁUDIA,Amazônia,2015-01-06 17:23:00


## Filtro de biomas e satélite de referência

In [5]:
df = data_loader.filter_reference(df)
df.shape

(1791186, 10)

## Série mensal por bioma

In [9]:
monthly = features.monthly_counts(df)
monthly.to_csv(PROCESSED_DIR / 'focos_mensais.csv', index=False)
monthly.tail()

,bioma,mes,focos
391,Pantanal,2025-08-01,48
392,Pantanal,2025-09-01,189
393,Pantanal,2025-10-01,241
394,Pantanal,2025-11-01,46
395,Pantanal,2025-12-01,75


## Exploração inicial dos dados


### Focos por bioma

In [10]:
df["bioma"].value_counts()

bioma
Amazônia    1034559
Cerrado      675683
Pantanal      80944
Name: count, dtype: int64

### Focos por ano

In [12]:
df.groupby(df["datahora"].dt.year).size()

datahora
2015    185990
2016    151778
2017    179974
2018    109485
2019    163075
2020    189096
2021    146076
2022    173555
2023    155932
2024    236312
2025     99913
dtype: int64

In [16]:
(df["bioma"].value_counts(normalize=True) * 100).round(1)

bioma
Amazônia    57.8
Cerrado     37.7
Pantanal     4.5
Name: proportion, dtype: float64

### Valores ausentes


In [14]:
df.isna().sum()

id_bdq       0
foco_id      0
lat          0
lon          0
data_pas     0
pais         0
estado       0
municipio    0
bioma        0
datahora     0
dtype: int64

In [15]:
df[df["datahora"].dt.year == 2025]["datahora"].dt.month.value_counts().sort_index()

datahora
1      1979
2      1048
3      1467
4       895
5      3373
6      5347
7      7525
8     14757
9     21943
10    19965
11    14818
12     6796
Name: count, dtype: int64

## Conclusões desta etapa

- **Volume:** 1.791.186 focos de queimadas detectados pelo satélite de referência entre 2015 e 2025 nos três biomas analisados.
- **Distribuição por bioma:** Amazônia (57,8%), Cerrado (37,7%) e Pantanal (4,5%). O Pantanal tem menos focos em números absolutos, mas é um bioma bem menor em área; comparar por área seria mais justo.
- **Evolução anual:** o total oscila entre cerca de 100 mil e 190 mil focos por ano, com exceção de 2024, que chegou a 236.312.
- **Qualidade dos dados:** não há valores ausentes em nenhuma coluna.
- **Pontos a investigar:** o que explica o pico de 2024 (hipótese: seca e clima, a testar com dados climáticos) e a queda em 2025, que tem os 12 meses mas é o ano com menos focos da série. Também vale conferir na documentação do INPE se houve mudança no sistema de detecção ao longo do período, para saber se a queda é do fogo ou da medição.